In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install "ultralytics==8.4.183" paddlepaddle paddleocr

import zipfile
zipfile.ZipFile('/content/drive/MyDrive/yolo_runs/plate-verification-v0.3.0.zip').extractall('/content/')
%cd /content/plate-verification
!ls

In [ ]:
import glob
found = glob.glob('/content/drive/MyDrive/yolo_runs/amira_model/**/best.pt', recursive=True)
print(found)
MODEL = found[0]

In [ ]:
!python tools/run_street.py --zip "/content/drive/MyDrive/streets/Rue_Bonaparte.zip" --name rue_bonaparte --model "{MODEL}" --out /content/out/rue_bonaparte

In [ ]:
!python tools/run_street.py --zip "/content/drive/MyDrive/streets/rue_de_four.zip" --name rue_de_four --model "{MODEL}" --out /content/out/rue_de_four

In [ ]:
import shutil, os, time
dest = '/content/drive/MyDrive/yolo_runs/pipeline_results'
os.makedirs(dest, exist_ok=True)
stamp = time.strftime('%Y%m%d_%H%M')
for n in ['rue_bonaparte', 'rue_de_four']:
    if os.path.exists(f'/content/out/{n}'):
        shutil.make_archive(f'{dest}/{n}_{stamp}', 'zip', f'/content/out/{n}')
        print('saved', n)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import datetime
import subprocess
import sys
import zipfile

base = Path('/content/drive/MyDrive/yolo_runs')
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
work = Path('/content') / f'recheck_{stamp}'

package = base / 'plate-verification-v0.3.1.zip'
assert package.exists(), f'Upload this file first: {package}'

with zipfile.ZipFile(package) as z:
    z.extractall(work)

repo = work / 'plate-verification'
saved = base / 'pipeline_results'
destination = saved / f'rechecked_{stamp}'

for street in ['rue_bonaparte', 'rue_de_four']:
    matches = list(saved.glob(f'{street}_*.zip'))
    if not matches:
        print(f'No saved ZIP found for {street}; skipping.')
        continue

    source = max(matches, key=lambda p: p.stat().st_mtime)
    print(f'\nRechecking {source.name}', flush=True)

    subprocess.run(
        [
            sys.executable, str(repo / 'tools/recheck_saved.py'),
            '--saved', str(source),
            '--out', str(destination / street),
        ],
        check=True,
    )

print('\nNew results saved in:', destination)

In [ ]:
%%writefile /content/ocr_compare.py
# OCR comparison on the saved plate crops (no YOLO, no training). Never deletes anything.
import io, re, sys, json, shutil, zipfile, subprocess, datetime
from pathlib import Path
import cv2, numpy as np, pandas as pd
from paddleocr import TextRecognition

BASE = Path(sys.argv[1])          # folder that contains rechecked_*/<street>/updated_results.zip
REPO = Path(sys.argv[2])          # unzipped plate-verification v0.3.1
OUT = Path(sys.argv[3])           # new output folder
OUT.mkdir(parents=True, exist_ok=False)
MODELS = ['PP-OCRv5_mobile_rec', 'en_PP-OCRv5_mobile_rec', 'PP-OCRv5_server_rec']
PREPS = ['as_is', 'big_contrast', 'big_inverted']

def digits_of(t):
    runs = re.findall(r'\d+', str(t or ''))
    if not runs:
        return ''
    b = runs[0].lstrip('0')
    return b if 1 <= len(b) <= 3 else ''

def prep(img, mode):
    if mode == 'as_is':
        return img
    big = cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_LANCZOS4)
    lab = cv2.cvtColor(big, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = cv2.createCLAHE(2.0, (4, 4)).apply(lab[:, :, 0])
    big = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    if mode == 'big_contrast':
        return big
    g = cv2.cvtColor(big, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(255 - g, cv2.COLOR_GRAY2BGR)

def field(res, key):
    try:
        return res[key]
    except Exception:
        j = getattr(res, 'json', None)
        j = j() if callable(j) else j
        return (j or {}).get('res', j or {}).get(key)

streets = {}
for name in ['rue_bonaparte', 'rue_de_four']:
    found = sorted(BASE.glob(f'rechecked_*/{name}/updated_results.zip'))
    if not found:
        print('missing', name); continue
    zz = zipfile.ZipFile(found[-1])
    det = pd.read_csv(zz.open('detections.csv'))
    res = pd.read_csv(zz.open('results/detections.csv'))
    plates = pd.read_csv(zz.open('results/plates.csv'))
    crops = pd.read_csv(zz.open('web/plate_crops.csv'))
    det['box'] = det.groupby('image').cumcount()
    det = det.merge(crops[['image', 'box', 'plate_crop']], on=['image', 'box'], how='left')
    det['plate_id'] = res['plate_id'].values
    exp = plates.set_index('plate_id')['expected_address'].astype(str).str.extract(r'^(\d+)')[0]
    det['expected'] = det['plate_id'].map(exp).fillna('')
    det['img'] = [cv2.imdecode(np.frombuffer(zz.read('web/' + p), np.uint8), cv2.IMREAD_COLOR) for p in det['plate_crop']]
    streets[name] = (found[-1], det)
    print(name, len(det), 'plates,', int((det.expected != '').sum()), 'with an address at their position')

rows = []
reads = {}
for model in MODELS:
    try:
        rec = TextRecognition(model_name=model)
    except Exception as e:
        print('SKIP', model, '->', str(e)[:200]); continue
    for name, (_, det) in streets.items():
        for mode in PREPS:
            texts, confs = [], []
            for img in det['img']:
                out = list(rec.predict(input=prep(img, mode), batch_size=1))
                raw = str(field(out[0], 'rec_text') or '') if out else ''
                sc = float(field(out[0], 'rec_score') or 0) if out else 0.0
                d = digits_of(raw)
                texts.append(d); confs.append(sc if d else 0.0)
            reads[(model, name, mode)] = (texts, confs)
    # ensemble: best confident reading over the 3 preprocessings
    for name, (_, det) in streets.items():
        T = np.array([reads[(model, name, m)][0] for m in PREPS], dtype=object)
        C = np.array([reads[(model, name, m)][1] for m in PREPS])
        k = C.argmax(axis=0)
        reads[(model, name, 'ensemble')] = (list(T[k, range(len(k))]), list(C[k, range(len(k))]))
    print('done', model, flush=True)

for (model, name, mode), (texts, confs) in reads.items():
    det = streets[name][1]
    t, c = np.array(texts, dtype=object), np.array(confs)
    has = det['expected'].values != ''
    rows.append(dict(model=model, prep=mode, street=name,
                     read_80=round(float(((t != '') & (c >= 0.8)).mean()), 3),
                     agree_with_address=round(float((t[has] == det['expected'].values[has]).mean()), 3)))
tab = pd.DataFrame(rows)
tab.to_csv(OUT / 'ocr_comparison.csv', index=False)
score = tab.groupby(['model', 'prep'])[['read_80', 'agree_with_address']].mean()
score['total'] = score.read_80 + score.agree_with_address
score = score.sort_values('total', ascending=False)
print('\n', score.round(3).to_string())
best = score.index[0]
print('\nBEST:', best)

for name, (zpath, det) in streets.items():
    texts, confs = reads[(best[0], name, best[1])]
    work = OUT / f'{name}_input'
    with zipfile.ZipFile(zpath) as zz:
        zz.extractall(work)
    new = pd.read_csv(work / 'detections.csv')
    new.to_csv(work / 'detections_before_ocr_update.csv', index=False)
    new['text'] = texts
    new['ocr_conf'] = np.round(confs, 4)
    new['source'] = f'yolo26n+{best[0]}+{best[1]}'
    new.to_csv(work / 'detections.csv', index=False)
    packed = shutil.make_archive(str(OUT / f'{name}_new_ocr'), 'zip', work)
    subprocess.run([sys.executable, str(REPO / 'tools/recheck_saved.py'), '--saved', packed,
                    '--out', str(OUT / f'{name}_result')], check=True)
json.dump({'best_model': best[0], 'best_prep': best[1]}, open(OUT / 'best.json', 'w'))
print('\nSaved in', OUT)

In [ ]:
!pip -q install paddlepaddle paddleocr
from google.colab import drive; drive.mount('/content/drive')
import glob, zipfile, datetime
repo_zip = sorted(glob.glob('/content/drive/MyDrive/**/plate-verification-v0.3.1.zip', recursive=True))[-1]
zipfile.ZipFile(repo_zip).extractall('/content/v031')
repo = glob.glob('/content/v031/**/tools/recheck_saved.py', recursive=True)[0].rsplit('/tools/', 1)[0]
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
out = f'/content/drive/MyDrive/yolo_runs/pipeline_results/ocr_test_{stamp}'
print('repo:', repo, '\nout:', out)
!python /content/ocr_compare.py /content/drive/MyDrive/yolo_runs/pipeline_results "{repo}" "{out}"

In [ ]:
%%writefile /content/ocr_compare2.py
# OCR comparison v2: EasyOCR (GPU) vs PaddleOCR on the saved plate crops. No YOLO, no training. Never deletes anything.
import re, sys, json, time, shutil, zipfile, subprocess
from pathlib import Path
import cv2, numpy as np, pandas as pd

BASE, REPO, OUT = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3])
OUT.mkdir(parents=True, exist_ok=False)
ENGINES = ['easyocr', 'PP-OCRv5_mobile_rec', 'en_PP-OCRv5_mobile_rec']
PREPS = ['as_is', 'big_contrast', 'big_inverted']

def digits_of(t):
    runs = re.findall(r'\d+', str(t or ''))
    if not runs:
        return ''
    b = runs[0].lstrip('0')
    return b if 1 <= len(b) <= 3 else ''

def prep(img, mode):
    if mode == 'as_is':
        return img
    big = cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_LANCZOS4)
    lab = cv2.cvtColor(big, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = cv2.createCLAHE(2.0, (4, 4)).apply(lab[:, :, 0])
    big = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    if mode == 'big_contrast':
        return big
    g = cv2.cvtColor(big, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(255 - g, cv2.COLOR_GRAY2BGR)

def field(res, key):
    try:
        return res[key]
    except Exception:
        j = getattr(res, 'json', None)
        j = j() if callable(j) else j
        return (j or {}).get('res', j or {}).get(key)

def make_reader(engine):
    if engine == 'easyocr':
        import easyocr
        reader = easyocr.Reader(['en'], gpu=True, verbose=False)
        def run(imgs):
            out = []
            for img in imgs:
                parts = reader.readtext(img, allowlist='0123456789', detail=1, paragraph=False)
                parts = sorted(parts, key=lambda p: min(pt[0] for pt in p[0]))
                text = ''.join(p[1] for p in parts)
                conf = min([float(p[2]) for p in parts], default=0.0)
                out.append((text, conf))
            return out
        return run
    from paddleocr import TextRecognition
    rec = TextRecognition(model_name=engine)
    def run(imgs):
        res = list(rec.predict(input=list(imgs), batch_size=32))
        return [(str(field(r, 'rec_text') or ''), float(field(r, 'rec_score') or 0)) for r in res]
    return run

streets = {}
for name in ['rue_bonaparte', 'rue_de_four']:
    found = sorted(BASE.glob(f'rechecked_*/{name}/updated_results.zip'))
    if not found:
        print('missing', name); continue
    zz = zipfile.ZipFile(found[-1])
    det = pd.read_csv(zz.open('detections.csv'))
    res = pd.read_csv(zz.open('results/detections.csv'))
    plates = pd.read_csv(zz.open('results/plates.csv'))
    crops = pd.read_csv(zz.open('web/plate_crops.csv'))
    det['box'] = det.groupby('image').cumcount()
    det = det.merge(crops[['image', 'box', 'plate_crop']], on=['image', 'box'], how='left')
    det['plate_id'] = res['plate_id'].values
    exp = plates.set_index('plate_id')['expected_address'].astype(str).str.extract(r'^(\d+)')[0]
    det['expected'] = det['plate_id'].map(exp).fillna('')
    det['img'] = [cv2.imdecode(np.frombuffer(zz.read('web/' + p), np.uint8), cv2.IMREAD_COLOR) for p in det['plate_crop']]
    streets[name] = (found[-1], det)
    print(name, len(det), 'plates', flush=True)

reads, rows = {}, []
for engine in ENGINES:
    t0 = time.time()
    try:
        run = make_reader(engine)
    except Exception as e:
        print('SKIP', engine, '->', str(e)[:200], flush=True); continue
    for name, (_, det) in streets.items():
        for mode in PREPS:
            imgs = [prep(i, mode) for i in det['img']]
            got = []
            for s in range(0, len(imgs), 100):
                got += run(imgs[s:s + 100])
                print(f'  {engine} {name} {mode}: {len(got)}/{len(imgs)}  ({time.time() - t0:.0f}s)', flush=True)
            texts = [digits_of(t) for t, _ in got]
            confs = [c if d else 0.0 for d, (_, c) in zip(texts, got)]
            reads[(engine, name, mode)] = (texts, confs)
            pd.DataFrame({'image': det['image'], 'raw': [t for t, _ in got], 'text': texts, 'conf': confs}) \
              .to_csv(OUT / f'reads_{engine}_{name}_{mode}.csv', index=False)
        T = np.array([reads[(engine, name, m)][0] for m in PREPS], dtype=object)
        C = np.array([reads[(engine, name, m)][1] for m in PREPS])
        k = C.argmax(axis=0)
        reads[(engine, name, 'ensemble')] = (list(T[k, range(len(k))]), list(C[k, range(len(k))]))
    print(f'DONE {engine} in {time.time() - t0:.0f}s', flush=True)

for (engine, name, mode), (texts, confs) in reads.items():
    det = streets[name][1]
    t, c = np.array(texts, dtype=object), np.array(confs)
    has = det['expected'].values != ''
    rows.append(dict(engine=engine, prep=mode, street=name,
                     read_80=round(float(((t != '') & (c >= 0.8)).mean()), 3),
                     agree_with_address=round(float((t[has] == det['expected'].values[has]).mean()), 3)))
tab = pd.DataFrame(rows)
tab.to_csv(OUT / 'ocr_comparison.csv', index=False)
score = tab.groupby(['engine', 'prep'])[['read_80', 'agree_with_address']].mean()
score['total'] = score.read_80 + score.agree_with_address
score = score.sort_values('total', ascending=False)
print('\n', score.round(3).to_string())
best = score.index[0]
print('\nBEST:', best, flush=True)

for name, (zpath, det) in streets.items():
    texts, confs = reads[(best[0], name, best[1])]
    work = OUT / f'{name}_input'
    with zipfile.ZipFile(zpath) as zz:
        zz.extractall(work)
    new = pd.read_csv(work / 'detections.csv')
    new.to_csv(work / 'detections_before_ocr_update.csv', index=False)
    new['text'] = texts
    new['ocr_conf'] = np.round(confs, 4)
    new['source'] = f'yolo26n+{best[0]}+{best[1]}'
    new.to_csv(work / 'detections.csv', index=False)
    packed = shutil.make_archive(str(OUT / f'{name}_new_ocr'), 'zip', work)
    subprocess.run([sys.executable, str(REPO / 'tools/recheck_saved.py'), '--saved', packed,
                    '--out', str(OUT / f'{name}_result')], check=True)
json.dump({'best_engine': best[0], 'best_prep': best[1]}, open(OUT / 'best.json', 'w'))
print('\nSaved in', OUT)

In [ ]:
!pip -q install paddlepaddle paddleocr easyocr
from google.colab import drive; drive.mount('/content/drive')
import glob, zipfile, datetime
repo_zip = sorted(glob.glob('/content/drive/MyDrive/**/plate-verification-v0.3.1.zip', recursive=True))[-1]
zipfile.ZipFile(repo_zip).extractall('/content/v031')
repo = glob.glob('/content/v031/**/tools/recheck_saved.py', recursive=True)[0].rsplit('/tools/', 1)[0]
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
out = f'/content/drive/MyDrive/yolo_runs/pipeline_results/ocr_test_{stamp}'
print('repo:', repo, '\nout:', out)
!python /content/ocr_compare2.py /content/drive/MyDrive/yolo_runs/pipeline_results "{repo}" "{out}"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
import subprocess
import sys
import zipfile

drive_root = Path("/content/drive/MyDrive")
archive = drive_root / "plate-verification-v0.3.2-corrected.zip"
model = drive_root / "yolo_runs/amira_model/best.pt"
street_dir = drive_root / "streets"

inputs = [
    ("rue_bonaparte", street_dir / "Rue_Bonaparte.zip"),
    ("rue_de_four", street_dir / "rue_de_four.zip"),
]

# Check inputs before installing or running anything.
for path in [archive, model, *(p for _, p in inputs)]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {path}")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
local = Path(f"/content/v032_corrected_{stamp}")
local.mkdir(exist_ok=False)

with zipfile.ZipFile(archive) as z:
    z.extractall(local)

repo = local / "plate-verification"
output_root = drive_root / f"yolo_runs/pipeline_results/v032_corrected_{stamp}"
output_root.mkdir(parents=True, exist_ok=False)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics", "paddlepaddle", "paddleocr"],
    check=True,
)

for name, street_zip in inputs:
    out = output_root / name
    print(f"\n===== Running {name} =====", flush=True)

    subprocess.run(
        [
            sys.executable, "-u", str(repo / "tools/run_street.py"),
            "--zip", str(street_zip),
            "--name", name,
            "--model", str(model),
            "--out", str(out),
            "--work", str(local / "street_inputs"),
            "--addresses", str(repo / "data/adresse_paris.csv"),
            "--config", str(repo / "config/numbers_only.toml"),
            "--imgsz", "1280",
        ],
        cwd=repo,
        check=True,
    )

print(f"\nFinished. Results saved in:\n{output_root}")

small OCR check

In [ ]:
import subprocess
import sys
from pathlib import Path
from datetime import datetime

# Locate the small diagnostic script.
script = Path("/content/ocr_check_20261024_192015/ocr_check/check.py")
if not script.is_file():
    raise FileNotFoundError(
        "The diagnostic script is missing. Stop here and send this message."
    )

# Install into the SAME Python environment used to run the script.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "paddlepaddle", "paddleocr"
    ],
    check=True,
)

# Confirm the required import works before starting.
subprocess.run(
    [
        sys.executable, "-c",
        "from paddleocr import TextRecognition; "
        "print('PaddleOCR import OK', flush=True)"
    ],
    check=True,
)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
out = Path(
    f"/content/drive/MyDrive/yolo_runs/ocr_diagnostic_{stamp}"
)

# Stream the actual output/error into the notebook.
command = [
    sys.executable, "-u", str(script),
    "--out", str(out),
]
with subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()

if code != 0:
    raise RuntimeError(
        "The diagnostic stopped. The actual error is printed directly above."
    )

print("\nFinished. Send this file here:")
print(out / "ocr_diagnostic.csv")

 test EasyOCR with its digits-only

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
import subprocess
import sys
import importlib
import zipfile

# Unpack the 30 reviewed crops from Drive into a NEW temporary folder.
archive = Path("/content/drive/MyDrive/ocr_check_30.zip")
if not archive.is_file():
    raise FileNotFoundError(
        "ocr_check_30.zip is not directly in My Drive. Stop and tell me where it is."
    )

stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
work = Path(f"/content/easyocr_check_{stamp}")
work.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as z:
    z.extractall(work)

folder = work / "ocr_check"
if not (folder / "labels.csv").is_file():
    raise FileNotFoundError("labels.csv is missing inside ocr_check_30.zip.")

# Install EasyOCR in this notebook's Python.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "easyocr"],
    check=True,
)
importlib.invalidate_caches()

import easyocr
import torch
import cv2
import pandas as pd

labels = pd.read_csv(folder / "labels.csv", dtype=str, keep_default_na=False)

gpu = torch.cuda.is_available()
print("Using GPU:", gpu, flush=True)
reader = easyocr.Reader(["en"], gpu=gpu)

rows = []
for _, row in labels.iterrows():
    image = cv2.imread(str(folder / row["crop"]), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise RuntimeError(f"Cannot open crop: {row['crop']}")
    h, w = image.shape

    # Read the whole crop, digits only.
    results = reader.recognize(
        image,
        horizontal_list=[[0, w, 0, h]],
        free_list=[],
        allowlist="0123456789",
        detail=1,
        paragraph=False,
    )

    text, confidence = "", 0.0
    if results:
        best = max(results, key=lambda r: float(r[2]))
        text, confidence = str(best[1]).strip(), float(best[2])

    rows.append({
        "item": row["item"],
        "street": row["street"],
        "image": row["image"],
        "human_reading": row["human_reading"],
        "previous_pipeline_reading": row["system_reading"],
        "easyocr_digits": text,
        "easyocr_confidence": confidence,
    })
    print(f"Checked {row['item']} / 30", flush=True)

out = Path("/content/drive/MyDrive/yolo_runs") / f"easyocr_digits_{stamp}"
out.mkdir(parents=True, exist_ok=False)
csv = out / "easyocr_digits_comparison.csv"
pd.DataFrame(rows).to_csv(csv, index=False)
print("\nFINISHED — send this CSV here:\n", csv)